In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

import sys
from pathlib import Path

sys.path.append(str(Path("../src").resolve()))   # folder that CONTAINS the .py file

import utils
from utils import clean, get_data, get_results       # for src/my_utils.py

## where trained models are saved (project-root models/ folder)
MODEL_DIR = Path("../models").resolve()
MODEL_DIR.mkdir(exist_ok=True)

In [2]:
df_2024 = pd.read_csv("../data/df_2024_final.csv").assign(season=2024)
df_2025 = pd.read_csv("../data/df_2025_final.csv").assign(season=2025)
df_2026 = pd.read_csv("../data/df_2026_final.csv").assign(season=2026)

# make 2026 consistent with 2024/2025: drop the 2026-only columns and use the same column order
# (2026 lacks the defensive columns, which are ~all NaN in 2025 and unused by the position frames -> filled with NaN)
df_2026 = df_2026.drop(columns=["pro_team_id", "season_type"], errors="ignore")
df_2026 = df_2026.reindex(columns=df_2024.columns)
df_2026 = df_2026.astype(df_2024.dtypes.to_dict(), errors="ignore")

df = pd.concat([df_2024, df_2025, df_2026], ignore_index=True)

In [3]:
df.head()
df= df.drop(columns="Unnamed: 0")

In [4]:

## necessary columns for each position to define separate data frames
id_cols = ['player_id', 'player', 'position', 'team', 'season', 'week',
           'date', 'opponent', 'atVs']

context_cols = ['opp_def_rank', 'team_pass_rank', 'team_rush_rank',
                'stadium', 'roof', 'surface', 'temp', 'wind', 'location',
                'gametime']

target_cols = ['fantasy_points', 'projected_points']



rushing = ['rushingAttempts', 'rushingYards', 'yardsPerRushAttempt',
           'rushingTouchdowns', 'longRushing']
receiving = ['receptions', 'receivingTargets', 'receivingYards',
             'yardsPerReception', 'receivingTouchdowns', 'longReception']
ball_security = ['fumbles', 'fumblesLost']
passing = ['completions', 'passingAttempts', 'passingYards', 'completionPct',
           'yardsPerPassAttempt', 'passingTouchdowns', 'interceptions',
           'longPassing', 'sacks', 'QBRating', 'adjQBR']
kicking = ['fieldGoalsMade', 'fieldGoalAttempts', 'fieldGoalPct',
           'longFieldGoalMade', 'fieldGoalsMadeYardsAverage',
           'fieldGoalsMade1_19', 'fieldGoalAttempts1_19',
           'fieldGoalsMade20_29', 'fieldGoalAttempts20_29',
           'fieldGoalsMade30_39', 'fieldGoalAttempts30_39',
           'fieldGoalsMade40_49', 'fieldGoalAttempts40_49',
           'fieldGoalsMade50', 'fieldGoalAttempts50',
           'extraPointsMade', 'extraPointAttempts', 'totalKickingPoints']

base = id_cols + context_cols + target_cols

position_cols = {
    'QB': base + passing + rushing + ball_security,
    'WR': base + receiving + rushing + ball_security,
    'TE': base + receiving + rushing + ball_security,
    'K':  base + kicking,
    "RB": base + rushing + receiving + ball_security
}
## generate each postional dataframe based on what scoring metrics contribute to their respective fantasy score
pos_dfs = {pos: df.loc[df['position'] == pos, cols].copy()
           for pos, cols in position_cols.items()}

qb_df, wr_df, te_df, k_df, rb_df = (pos_dfs[p] for p in ['QB', 'WR', 'TE', 'K', 'RB'])

In [5]:
## combined data frames based on cluster
qb_te = pd.concat([qb_df, te_df])
#qb_te.head()


In [6]:
wr_k = pd.concat([wr_df, k_df])

In [7]:
# now we need to sort each data frame in order
rb_df = rb_df.sort_values(['player_id', 'season', 'week']).reset_index(drop=True)
qb_te = qb_te.sort_values(['player_id', 'season', 'week']).reset_index(drop=True)
wr_k = wr_k.sort_values(['player_id', 'season', 'week']).reset_index(drop=True)


In [8]:
## need to shift by -1 to create target of next weeks fantasy score
g_1 = rb_df.groupby(['player_id', "season"])
rb_df["target_fantasy_points"] = g_1["fantasy_points"].shift(-1)
rb_df['next_week_opponent'] = g_1['opponent'].shift(-1)
rb_df['next_week_atVs'] = g_1['atVs'].shift(-1)
rb_df['next_week_projected'] = g_1['projected_points'].shift(-1) 

g_2 = qb_te.groupby(['player_id', "season"])
qb_te["target_fantasy_points"] = g_2["fantasy_points"].shift(-1)
qb_te['next_week_opponent'] = g_2['opponent'].shift(-1)
qb_te['next_week_atVs'] = g_2['atVs'].shift(-1)
qb_te['next_week_projected'] = g_2['projected_points'].shift(-1) 

g_3 = wr_k.groupby(['player_id', "season"])
wr_k["target_fantasy_points"] = g_3["fantasy_points"].shift(-1)
wr_k["next_week_opponent"] = g_3["opponent"].shift(-1)
wr_k["next_week_atVs"] = g_3["atVs"].shift(-1)
wr_k["next_week_projected"] = g_3["projected_points"].shift(-1)
for col in context_cols:
    rb_df[f"next_week_{col}"] = g_1[col].shift(-1)
    qb_te[f"next_week_{col}"] = g_2[col].shift(-1)
    wr_k[f"next_week_{col}"] = g_3[col].shift(-1)


## drop the last week of each season: it has no next week so its targets are all NaN,
## and the previous week already saw that week's data through the shift (redundant / leakage)
## (for 2026 this is the latest week in the data, which hasn't been played yet)
rb_df = rb_df[rb_df["week"] < rb_df.groupby("season")["week"].transform("max")].reset_index(drop=True)
qb_te = qb_te[qb_te["week"] < qb_te.groupby("season")["week"].transform("max")].reset_index(drop=True)
wr_k = wr_k[wr_k["week"] < wr_k.groupby("season")["week"].transform("max")].reset_index(drop=True)

In [9]:
## encode categorical columns


shared_cat = ['player','position', 'next_week_opponent', 'next_week_atVs', 'next_week_surface',
                    'next_week_roof', 'next_week_location', 'next_week_stadium', 'next_week_gametime', "team"]
target = "target_fantasy_points"

rb_df[shared_cat] = rb_df[shared_cat].astype("category")
qb_te[shared_cat] = qb_te[shared_cat].astype("category")
wr_k[shared_cat] = wr_k[shared_cat].astype("category")

In [10]:
# generate training splits for each group
# train: 2024 + 2025 weeks 1-9, validation: 2025 weeks 10+, test: 2026
train_data_1 = rb_df[(rb_df["season"] == 2024) | ((rb_df["season"] == 2025) & (rb_df["week"] < 10))]
train_data_2 = qb_te[(qb_te["season"] == 2024) | ((qb_te["season"] == 2025) & (qb_te["week"] < 10))]
train_data_3 = wr_k[(wr_k["season"] == 2024) | ((wr_k["season"] == 2025) & (wr_k["week"] < 10))]

val_1 = rb_df[(rb_df["season"] == 2025) & (rb_df["week"] >= 10)]
val_2 = qb_te[(qb_te["season"] == 2025) & (qb_te["week"] >= 10)]
val_3 = wr_k[(wr_k["season"] == 2025) & (wr_k["week"] >= 10)]

test_1 = rb_df[rb_df["season"] == 2026]
test_2 = qb_te[qb_te["season"] == 2026]
test_3 = wr_k[wr_k["season"] == 2026]


In [11]:
## common columns across all the data frames
common = list(set(rb_df.columns) & set(qb_te.columns) & set(wr_k.columns))
irrelevant_columns = [
    'player_id', 'opponent', "date",
    'eventId', 'game_id', 'gameday', 'gametime', 'stadium', 'result',
    'score', 'fantasy_points', 'target_next_week_fantasy', 'espn',
    '_set', 'season',
    'med_proj_pts',
    'projected_points',
    'atVs', 'roof', 'surface', 'temp', 'wind', 'location',
    'opp_def_rank', 'team_pass_rank', 'team_rush_rank', 'next_week_projected',
]

# want to get all irrelevant columns that dont contribute to score
irrelevant = list(set(common) & set(irrelevant_columns)) + ["target_fantasy_points"]



In [12]:


## Running back only
X1_train, y1_train = get_data(train_data_1, irrelevant, target)
X2_train, y2_train = get_data(train_data_2, irrelevant, target)
X3_train, y3_train = get_data(train_data_3, irrelevant, target)

##quarter back/ tight end
X1_val, y1_val = get_data(val_1, irrelevant, target)
X2_val, y2_val = get_data(val_2, irrelevant, target)
X3_val, y3_val = get_data(val_3, irrelevant, target)
## Wide reciever/kicker
X1_test, y1_test = get_data(test_1, irrelevant, target)
X2_test, y2_test = get_data(test_2, irrelevant, target)
X3_test, y3_test = get_data(test_3, irrelevant, target)

In [13]:
model1_path = MODEL_DIR / "xgb_rb.json"   # RB model, saved after training
model1 = xgb.XGBRegressor(
    tree_method = "hist",
    n_estimators=10000, #Higher is better
    learning_rate=0.005, #lower is better
    max_depth=5, #higher is better
    min_child_weight=3, #mid:1-10
    subsample=0.7, #mid:0-1
    colsample_bytree=0.7, #mid:0-1
    random_state=12, #seed
    enable_categorical=True,
    early_stopping_rounds=100     # stops when validation stops improving (old run overfit after ~50 trees)
)



In [14]:
model1.fit(
    X1_train, y1_train,
    eval_set=[(X1_val, y1_val)],
    verbose=50 #how often to print validation score 
)
model1.save_model(str(model1_path))
print(f"saved model1 to {model1_path}")

[0]	validation_0-rmse:8.08507
[50]	validation_0-rmse:7.36892
[100]	validation_0-rmse:6.88682
[150]	validation_0-rmse:6.58586
[200]	validation_0-rmse:6.39093
[250]	validation_0-rmse:6.27249
[300]	validation_0-rmse:6.20677
[350]	validation_0-rmse:6.17751
[400]	validation_0-rmse:6.15912
[450]	validation_0-rmse:6.15027
[500]	validation_0-rmse:6.15356
[550]	validation_0-rmse:6.16282
[561]	validation_0-rmse:6.16432
saved model1 to /home/tristan/Documents/MSDS/CSCI 5502/CSCI_5502_Fantasy_Project/models/xgb_rb.json


In [15]:
model2_path = MODEL_DIR / "xgb_qb_te.json"   # QB/TE model, saved after training
model2 = xgb.XGBRegressor(
    tree_method = "hist",
    n_estimators=10000, #Higher is better
    learning_rate=0.005, #lower is better
    max_depth=5, #higher is better
    min_child_weight=3, #mid:1-10
    subsample=0.7, #mid:0-1
    colsample_bytree=0.7, #mid:0-1
    random_state=12, #seed
    enable_categorical=True,
    early_stopping_rounds=100      # stops when validation stops improving (old run overfit after ~50 trees)
)
model2.fit(X2_train, y2_train,
        eval_set=[(X2_val, y2_val)],
        verbose=50)
model2.save_model(str(model2_path))
print(f"saved model2 to {model2_path}")

[0]	validation_0-rmse:7.98945
[50]	validation_0-rmse:7.18448
[100]	validation_0-rmse:6.65362
[150]	validation_0-rmse:6.31661
[200]	validation_0-rmse:6.11840
[250]	validation_0-rmse:6.00409
[300]	validation_0-rmse:5.94049
[350]	validation_0-rmse:5.92049
[400]	validation_0-rmse:5.91549
[450]	validation_0-rmse:5.92211
[500]	validation_0-rmse:5.93609
[505]	validation_0-rmse:5.93647
saved model2 to /home/tristan/Documents/MSDS/CSCI 5502/CSCI_5502_Fantasy_Project/models/xgb_qb_te.json


In [16]:
model3_path = MODEL_DIR / "xgb_wr_k.json"   # WR/K model, saved after training
model3 = xgb.XGBRegressor(
    tree_method = "hist",
    n_estimators=10000, #Higher is better
    learning_rate=0.005, #lower is better
    max_depth=5, #higher is better
    min_child_weight=3, #mid:1-10
    subsample=0.7, #mid:0-1
    colsample_bytree=0.7, #mid:0-1
    random_state=12, #seed
    enable_categorical=True,
    early_stopping_rounds=100      # stops when validation stops improving (old run overfit after ~50 trees)
)
model3.fit(X3_train, y3_train,
        eval_set=[(X3_val, y3_val)],
        verbose=50)
model3.save_model(str(model3_path))
print(f"saved model3 to {model3_path}")

[0]	validation_0-rmse:6.96824
[50]	validation_0-rmse:6.47113
[100]	validation_0-rmse:6.17618
[150]	validation_0-rmse:5.99242
[200]	validation_0-rmse:5.88609
[250]	validation_0-rmse:5.82578
[300]	validation_0-rmse:5.80372
[350]	validation_0-rmse:5.80595
[400]	validation_0-rmse:5.81077
[419]	validation_0-rmse:5.81341
saved model3 to /home/tristan/Documents/MSDS/CSCI 5502/CSCI_5502_Fantasy_Project/models/xgb_wr_k.json


In [17]:
## get our results for model 1
rb_results = get_results(rb_df, X1_test, y1_test, model1)

Model: MAE=4.544  RMSE=6.166  R2=0.392
ESPN: MAE=3.818  RMSE=5.517  R2=0.513


In [18]:
# model 2 results
qb_te_results=get_results(qb_te, X2_test, y2_test, model2)

Model: MAE=4.634  RMSE=6.071  R2=0.447
ESPN: MAE=3.677  RMSE=5.372  R2=0.567


In [19]:
wr_k_results = get_results(wr_k, X3_test, y3_test, model3)

Model: MAE=4.657  RMSE=6.154  R2=0.297
ESPN: MAE=4.038  RMSE=5.593  R2=0.420


In [20]:
all_pos = pd.concat([rb_results, qb_te_results, wr_k_results], ignore_index=True)
position_table = pd.crosstab(all_pos['position'], all_pos['winner'], margins=True)
position_table

winner,Model,espn,All
position,,,
K,124,112,236
QB,96,190,286
RB,298,442,740
TE,265,485,750
WR,423,728,1151
All,1206,1957,3163


## Player lookup
Enter a player's name to compare the model's predictions with their actual fantasy points and ESPN's projection for each test week (2026). `predicted_week` is the player's next game after the week the features come from (bye weeks are skipped).

In [21]:
# def next_game_weeks(player, season):
#     """Map each week a player played to the week of their next game (skips bye weeks)."""
#     weeks = sorted(df.loc[(df["player"] == player) & (df["season"] == season), "week"].unique())
#     return dict(zip(weeks, weeks[1:]))

# def player_report(name, results=all_pos, plot=True):
#     """Show model vs ESPN vs actual for one player (case-insensitive, partial names OK)."""
#     matches = results[results["player"].str.contains(name, case=False, na=False, regex=False)]
#     if matches.empty:
#         print(f"No test rows found for '{name}'.")
#         return None
#     players = matches["player"].unique()
#     if len(players) > 1:
#         print(f"Multiple players match '{name}': {list(players)}. Be more specific.")
#         return None

#     p = matches.sort_values(["season", "week"]).copy()
#     p["predicted_week"] = [next_game_weeks(players[0], s).get(w) for s, w in zip(p["season"], p["week"])]
#     cols = ["player", "position", "season", "predicted_week", "next_week_opponent",
#             "fantasy_points", "model_prediction", "espn_projected",
#             "model_error_abs", "espn_error_abs", "winner"]
#     table = p[cols].rename(columns={"fantasy_points": "actual", "next_week_opponent": "opponent"}).round(2)

#     print(f"{players[0]} ({p['position'].iloc[0]}): {len(p)} test weeks")
#     print(f"  Model MAE: {p['model_error_abs'].mean():.2f}   ESPN MAE: {p['espn_error_abs'].mean():.2f}")
#     print(f"  Model wins {(p['winner'] == 'Model').sum()} / {len(p)} weeks")

#     if plot:
#         fig, ax = plt.subplots(figsize=(9, 4))
#         ax.plot(table["predicted_week"], table["actual"], marker="o", label="Actual")
#         ax.plot(table["predicted_week"], table["model_prediction"], marker="s", label="Model")
#         ax.plot(table["predicted_week"], table["espn_projected"], marker="^", linestyle="--", label="ESPN")
#         ax.set_xlabel("Week")
#         ax.set_ylabel("Fantasy points")
#         ax.set_title(f"{players[0]}: predicted vs actual")
#         ax.legend()
#         plt.show()
#     return table.reset_index(drop=True)

# player_report("Christian McCaffrey")

## Modified Groups

Based on the findings in the ```Modeling_data_prep.ipynb``` we found the clusters with the most similar coefficient vectors. However, the cluster of Wide Reciever and Kicker makes no sense in terms of the game of football and based on our Spearman coefficients, Wide reciever was also close to QB. I would like to explore the potential cluster of WR, TE, WR and leave RB and K in their own seperate model.

In [22]:
# #sort values to apply shifts
# df1 = df.sort_values(['player_id', 'season', 'week']).reset_index(drop=True)
# # shift data
# g = df1.groupby(['player_id', "season"])
# df1["target_fantasy_points"] = g["fantasy_points"].shift(-1)
# df1['next_week_opponent'] = g['opponent'].shift(-1)
# df1['next_week_atVs'] = g['atVs'].shift(-1)
# df1['next_week_projected'] = g['projected_points'].shift(-1)

# # drop the last week of each season (no next week -> NaN targets, already seen by the previous week's shift)
# df1 = df1[df1["week"] < df1.groupby("season")["week"].transform("max")].reset_index(drop=True)

In [23]:


#already have model for running back so we are going to leace that alone
# qb, wr, te, k = df1[df1["position"] == "QB"], df1[df1["position"] == "WR"], df1[df1["position"] == "TE"], df1[df1["position"] == "K"]

# df2 = pd.concat([qb, wr, te])

# df2[shared_cat] = df2[shared_cat].astype("category")
# k[shared_cat] = k[shared_cat].astype("cateogory")

# training1 = df2[(df2["season"] == 2024) | ((df2["season"] == 2025) & (df2["week"] < 10))]
# val1 = df2[(df2["season"] == 2025) & (df2["week"] >= 10)]
# test1 = df2[df2["season"] == 2026]

# k_train = k[(k["season"] == 2024)|((k["season"]==2025) & (k["week"] < 10))]
# k_val = k[(k["season"] == 2025) & (k["week"] >= 10)]
# k_test = k[k["season"] == 2026]

# train_x, train_y = get_data(training1, irrelevant, target)
# val_x, val_y = get_data(val1, irrelevant, target)
# test_x, test_y = get_data(test1, irrelevant, target)

# train_k_x, train_k_y = get_data(k_train, irrelevant, target)
# val_k_x, val_k_y = get_data(k_val, irrelevant, target)
# test_k_x, test_k_y = get_data(k_test, irrelevant, target)

# # df2.head()

